# Model Comparison

Evaluate checkpoints trained with the same dataset split as notebook 01 and compare validation Dice, IoU, loss, epochs, and prediction overlays. Run the training notebooks first.


In [ ]:
from pathlib import Path
import csv
import sys

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import SegmentationDataset, find_image_mask_pairs, split_pairs
from src.models import build_model
from src.train import validate

DATA_DIR = PROJECT_ROOT / "notebooks" / "data" / "Kvasir-SEG"
if not DATA_DIR.exists():
    DATA_DIR = PROJECT_ROOT / "data" / "Kvasir-SEG"

RUNS_DIRS = [PROJECT_ROOT / "notebooks" / "runs"]
MODEL_NAMES = ["unet", "attention_unet", "unet_plus_plus", "transunet"]
VAL_RATIO = 0.2
MAX_SAMPLES = None
SEED = 42
BATCH_SIZE = 4
IMAGE_SIZE = 256

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Device:", device)


## Compare training logs

This summarizes the best row from each available `training_log.csv`.


In [ ]:
def find_run_dir(model_name, required_file="best.pt"):
    for runs_dir in RUNS_DIRS:
        run_dir = runs_dir / model_name
        print(f"Checking {run_dir}...")
        if (run_dir / required_file).exists():
            return run_dir
    return None

def read_history(log_path):
    with log_path.open("r", encoding="utf-8") as file:
        rows = list(csv.DictReader(file))
    for row in rows:
        for key in ["epoch", "train_loss", "val_loss", "val_dice", "val_iou", "best_val_dice"]:
            if key in row and row[key] != "":
                row[key] = float(row[key])
    return rows

summary = []
histories = {}
for model_name in MODEL_NAMES:
    run_dir = find_run_dir(model_name, required_file="best.pt")
    if run_dir is None:
        print(f"Missing checkpoint for {model_name}. Train it first.")
        continue

    log_path = run_dir / "training_log.csv"
    if not log_path.exists():
        print(f"Found checkpoint for {model_name}, but no training_log.csv. Evaluation will still work.")
        continue

    history = read_history(log_path)
    histories[model_name] = history
    best_row = max(history, key=lambda row: row["val_dice"])
    summary.append({
        "model": model_name,
        "best_epoch": int(best_row["epoch"]),
        "epochs_run": len(history),
        "best_dice": best_row["val_dice"],
        "best_iou": best_row["val_iou"],
        "best_val_loss": best_row["val_loss"],
        "run_dir": str(run_dir),
    })

if summary:
    header = f'{"model":<18} {"best_epoch":>10} {"epochs":>8} {"dice":>8} {"iou":>8} {"val_loss":>10}'
    print(header)
    print("-" * len(header))
    for row in sorted(summary, key=lambda item: item["best_dice"], reverse=True):
        print(f'{row["model"]:<18} {row["best_epoch"]:>10} {row["epochs_run"]:>8} {row["best_dice"]:>8.4f} {row["best_iou"]:>8.4f} {row["best_val_loss"]:>10.4f}')
else:
    print("No training logs found. This is OK if the notebooks saved checkpoints only; run the evaluation cell next.")


In [ ]:
if summary:
    ordered = sorted(summary, key=lambda item: item["best_dice"], reverse=True)
    labels = [row["model"] for row in ordered]
    dice_values = [row["best_dice"] for row in ordered]
    iou_values = [row["best_iou"] for row in ordered]

    x = range(len(labels))
    plt.figure(figsize=(8, 4))
    plt.bar([v - 0.2 for v in x], dice_values, width=0.4, label="Dice")
    plt.bar([v + 0.2 for v in x], iou_values, width=0.4, label="IoU")
    plt.xticks(list(x), labels, rotation=15)
    plt.ylim(0, 1)
    plt.title("Best validation metrics")
    plt.legend()
    plt.tight_layout()
    plt.show()


## Evaluate best checkpoints

This reloads each `best.pt` checkpoint and evaluates it on the same validation split.


In [ ]:
pairs = find_image_mask_pairs(DATA_DIR)
_, val_pairs = split_pairs(pairs, val_ratio=VAL_RATIO, seed=SEED, max_samples=MAX_SAMPLES)
bce_loss = nn.BCEWithLogitsLoss()

evaluation = []
loaded_models = {}
for model_name in MODEL_NAMES:
    run_dir = find_run_dir(model_name, required_file="best.pt")
    if run_dir is None:
        print(f"Missing checkpoint for {model_name}. Train it first.")
        continue

    checkpoint_path = run_dir / "best.pt"
    checkpoint = torch.load(checkpoint_path, map_location=device)
    image_size = checkpoint.get("image_size", 128)
    base_channels = checkpoint.get("base_channels", 32)
    dataset = SegmentationDataset(val_pairs, image_size=image_size, augment=False)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    model = build_model(checkpoint["model_name"], base_channels=base_channels).to(device)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()

    val_loss, val_dice, val_iou = validate(model, loader, bce_loss, device)
    evaluation.append({"model": model_name, "loss": val_loss, "dice": val_dice, "iou": val_iou, "checkpoint": str(checkpoint_path)})
    loaded_models[model_name] = {"model": model, "image_size": image_size}

if evaluation:
    header = f'{"model":<18} {"loss":>10} {"dice":>8} {"iou":>8}'
    print(header)
    print("-" * len(header))
    for row in sorted(evaluation, key=lambda item: item["dice"], reverse=True):
        print(f'{row["model"]:<18} {row["loss"]:>10.4f} {row["dice"]:>8.4f} {row["iou"]:>8.4f}')
else:
    print("No checkpoints evaluated.")


## Prediction overlay

Compare predictions from available checkpoints on one validation image.


In [ ]:
if loaded_models:
    sample_pair = val_pairs[0]
    columns = 2 + len(loaded_models)
    plt.figure(figsize=(4 * columns, 4))

    reference_size = next(iter(loaded_models.values()))["image_size"]
    reference_dataset = SegmentationDataset([sample_pair], image_size=reference_size, augment=False)
    image, mask = reference_dataset[0]
    image_np = image.permute(1, 2, 0).numpy()
    mask_np = mask.squeeze(0).numpy()

    plt.subplot(1, columns, 1)
    plt.imshow(image_np)
    plt.title("Image")
    plt.axis("off")

    plt.subplot(1, columns, 2)
    plt.imshow(mask_np, cmap="gray")
    plt.title("True mask")
    plt.axis("off")

    col = 3
    for model_name, item in loaded_models.items():
        dataset = SegmentationDataset([sample_pair], image_size=item["image_size"], augment=False)
        image, _ = dataset[0]
        with torch.no_grad():
            logits = item["model"](image.unsqueeze(0).to(device))
            pred = (torch.sigmoid(logits)[0, 0].cpu() > 0.5).float().numpy()

        plt.subplot(1, columns, col)
        plt.imshow(image.permute(1, 2, 0).numpy())
        plt.imshow(pred, cmap="Reds", alpha=0.45)
        plt.title(model_name)
        plt.axis("off")
        col += 1

    plt.tight_layout()
    plt.show()
else:
    print("No loaded models to visualize.")


## Streamlit GUI

These cells add a small Streamlit app. Upload one image and compare the predicted disease mask from each trained checkpoint in `notebooks/runs/<model>/best.pt`.


In [ ]:
from pathlib import Path

APP_PATH = PROJECT_ROOT / "notebooks" / "streamlit_model_viewer.py"
APP_CODE = r'''from pathlib import Path
import sys

import numpy as np
import streamlit as st
import torch
from PIL import Image

APP_PATH = Path(__file__).resolve()
PROJECT_ROOT = APP_PATH.parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models import build_model

RUNS_DIR = PROJECT_ROOT / "notebooks" / "runs"
MODEL_NAMES = ["unet", "attention_unet", "unet_plus_plus", "transunet"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

st.set_page_config(page_title="Kvasir-SEG Model Viewer", layout="wide")
st.title("Kvasir-SEG Model Viewer")
st.caption("Upload one image and compare where each trained model predicts the polyp region.")

@st.cache_resource
def load_checkpoint(model_name):
    checkpoint_path = RUNS_DIR / model_name / "best.pt"
    if not checkpoint_path.exists():
        return None, checkpoint_path, None

    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model = build_model(
        checkpoint["model_name"],
        base_channels=checkpoint.get("base_channels", 32),
    ).to(DEVICE)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    return model, checkpoint_path, checkpoint

def preprocess(image, image_size):
    resized = image.convert("RGB").resize((image_size, image_size), Image.BILINEAR)
    array = np.asarray(resized, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(array).permute(2, 0, 1).unsqueeze(0)
    return resized, tensor

def predict(model, tensor, threshold):
    with torch.no_grad():
        logits = model(tensor.to(DEVICE))
        probs = torch.sigmoid(logits)[0, 0].cpu().numpy()
    mask = probs > threshold
    return probs, mask

def make_overlay(image, mask, alpha):
    base = np.asarray(image.convert("RGB"), dtype=np.float32)
    red = np.zeros_like(base)
    red[..., 0] = 255.0
    overlay = base.copy()
    overlay[mask] = (1.0 - alpha) * base[mask] + alpha * red[mask]
    return overlay.astype(np.uint8)

def resize_mask(mask, size):
    mask_image = Image.fromarray((mask.astype(np.uint8) * 255), mode="L")
    mask_image = mask_image.resize(size, Image.NEAREST)
    return np.asarray(mask_image) > 0

def make_voting_overlay(image, masks, alpha):
    base = np.asarray(image.convert("RGB"), dtype=np.float32)
    votes = np.stack(masks, axis=0).sum(axis=0).astype(np.float32)
    vote_ratio = votes / max(1, len(masks))

    heat = np.zeros_like(base)
    heat[..., 0] = 255.0
    heat[..., 1] = 190.0 * (1.0 - vote_ratio)

    strength = (vote_ratio * alpha)[..., None]
    overlay = base * (1.0 - strength) + heat * strength
    return overlay.astype(np.uint8), votes

available_models = []
missing_models = []
for model_name in MODEL_NAMES:
    checkpoint_path = RUNS_DIR / model_name / "best.pt"
    if checkpoint_path.exists():
        available_models.append(model_name)
    else:
        missing_models.append(model_name)

with st.sidebar:
    st.header("Settings")
    selected_models = st.multiselect(
        "Models",
        MODEL_NAMES,
        default=available_models,
    )
    threshold = st.slider("Mask threshold", 0.05, 0.95, 0.50, 0.05)
    alpha = st.slider("Overlay strength", 0.10, 0.90, 0.55, 0.05)
    st.write("Device:", str(DEVICE))
    st.write("Runs folder:", str(RUNS_DIR))

if missing_models:
    st.warning("Missing checkpoints: " + ", ".join(missing_models))

uploaded_file = st.file_uploader("Upload an endoscopy image", type=["jpg", "jpeg", "png", "bmp"])
if uploaded_file is None:
    st.info("Upload an image to see predictions from the trained models.")
    st.stop()

image = Image.open(uploaded_file).convert("RGB")
st.subheader("Input image")
st.image(image, caption="Uploaded image", use_container_width=True)

if not selected_models:
    st.warning("Select at least one model from the sidebar.")
    st.stop()

vote_masks = []
used_model_names = []
original_size = image.size

for model_name in selected_models:
    model, checkpoint_path, checkpoint = load_checkpoint(model_name)
    if model is None:
        st.error(f"{model_name}: checkpoint not found at {checkpoint_path}")
        continue

    image_size = checkpoint.get("image_size", 256)
    resized, tensor = preprocess(image, image_size)
    probs, mask = predict(model, tensor, threshold)
    overlay = make_overlay(resized, mask, alpha)
    vote_masks.append(resize_mask(mask, original_size))
    used_model_names.append(model_name)

    st.markdown(f"### {model_name}")
    st.caption(f"Checkpoint: {checkpoint_path} | image_size={image_size} | threshold={threshold:.2f}")

    col1, col2, col3 = st.columns(3)
    with col1:
        st.image(resized, caption="Model input", use_container_width=True)
    with col2:
        st.image(probs, caption="Probability map", clamp=True, use_container_width=True)
    with col3:
        st.image(overlay, caption="Predicted mask overlay", use_container_width=True)

    st.write(
        {
            "mean_probability": float(probs.mean()),
            "max_probability": float(probs.max()),
            "predicted_area_ratio": float(mask.mean()),
        }
    )

if vote_masks:
    st.markdown("## Voting overlay")
    st.caption(
        "Each pixel is stronger when more selected models predict it as disease. "
        "Light color means weak agreement; strong red means high agreement."
    )
    voting_overlay, vote_count = make_voting_overlay(image, vote_masks, alpha)

    col1, col2 = st.columns(2)
    with col1:
        st.image(voting_overlay, caption="Voting overlay", use_container_width=True)
    with col2:
        st.image(vote_count, caption="Vote count map", clamp=True, use_container_width=True)

    agreement = {
        f"pixels_with_{votes}_votes": float((vote_count == votes).mean())
        for votes in range(1, len(vote_masks) + 1)
    }
    st.write({"models_used": used_model_names, **agreement})
'''
APP_PATH.write_text(APP_CODE, encoding="utf-8")
print("Streamlit app written to:", APP_PATH)


In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("streamlit") is None:
    print("Streamlit is not installed. Run this once, then rerun this cell:")
    print(f"{sys.executable} -m pip install streamlit")
else:
    print("Starting Streamlit. Stop the cell/kernel when you want to close the app.")
    subprocess.run(
        [sys.executable, "-m", "streamlit", "run", str(APP_PATH)],
        cwd=PROJECT_ROOT,
        check=True,
    )
